# ENSF617 Main Training Notebook

This notebook mirrors the repository's top-level training workflow through the stable facade in `main.py`.
That facade now delegates most orchestration to the smaller `src/workflows/` modules, but notebook usage intentionally stays aligned with the script entrypoint.

**Purpose:** let notebook users prepare config, inspect runtime resolution, launch training, and inspect artifacts without re-implementing orchestration logic in notebook cells.

**Context:** the model, data, environment-resolution, and observability behavior still live in the same typed project modules used by the CLI path. The notebook is only a thin interactive surface over that shared workflow.

**How to use it:** edit the configuration cell, run the config-building cell, then run the workflow cell.

**Important disclaimer:** this notebook is intended for research and development use inside the repository. The default configuration is a baseline starting point, not a claim of optimal performance or clinical validity.

In [ ]:
# Notebook bootstrap.
#
# Purpose:
# make the repository's shared entrypoint and typed modules importable
# from a notebook session without duplicating path-bootstrap logic later.
#
# Context:
# - assume the notebook is launched from the repository root
# - add the repo root to `sys.path` so `defaults.py`, `main.py`, and the
#   `src/` package facade modules resolve the same way they do elsewhere
# - reuse the same stable facade that the CLI path exposes rather than
#   importing deep workflow internals directly in the notebook
from pathlib import Path
import sys

ROOT_DIR = Path.cwd()
assert (ROOT_DIR / "main.py").exists(), "Run this notebook from the repository root."
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from defaults import (
    DEFAULT_AZT1D_URL,
    build_default_config,
    build_default_observability_config,
    build_default_snapshot_config,
    build_default_train_config,
)
from config import Config
from environment import (
    collect_runtime_diagnostics,
    detect_runtime_environment,
    format_runtime_diagnostics,
    resolve_device_profile,
)
from main import (
    _apply_early_apple_silicon_environment_defaults,
    run_training_workflow,
)


In [ ]:
# Edit these values for your run.
#
# Purpose:
# keep the most common experiment knobs easy to tweak from one cell.
#
# Context:
# these remain plain notebook variables rather than one large nested
# dictionary so interactive experimentation stays lightweight and each
# changed setting remains obvious during review.
output_dir = ROOT_DIR / "artifacts" / "notebook_run"
dataset_url = DEFAULT_AZT1D_URL
processed_dir = ROOT_DIR / "data" / "processed"
processed_file_name = "azt1d_processed.csv"

# Main data-window and loader controls.
encoder_length = 168
prediction_length = 12
batch_size = 64
num_workers = 0
pin_memory = None
persistent_workers = None

# Main runtime/profile controls.
max_epochs = 20
device_profile = "auto"
accelerator = "auto"
devices = "auto"
precision = 32
enable_progress_bar = None
enable_rich_progress_bar = None
enable_device_stats = None
fail_on_preflight_errors = True

# Main optimizer/run-behavior controls.
learning_rate = 1e-3
weight_decay = 0.0
optimizer_name = "adam"
seed = 42

# Held-out evaluation/prediction toggles.
skip_test = False
skip_predict = False
save_predictions = True


In [ ]:
# Convert the editable notebook variables into the typed config objects
# used by the rest of the codebase.
#
# Purpose:
# make the notebook's flat variables explicit as typed project config
# objects before launching a longer run.
#
# Context:
# this cell mirrors the reusable Python workflow path in `main.py`,
# including the early Apple Silicon bootstrap step that needs to run
# before deeper Torch runtime setup in MPS environments.
#
# Responsibility boundary:
# this cell owns config construction and runtime-profile resolution; the
# later workflow cell owns actually fitting, testing, and exporting run
# artifacts.
config = build_default_config(
    dataset_url=dataset_url,
    raw_dir=ROOT_DIR / "data" / "raw",
    cache_dir=ROOT_DIR / "data" / "cache",
    extracted_dir=ROOT_DIR / "data" / "extracted",
    processed_dir=processed_dir,
    processed_file_name=processed_file_name,
    encoder_length=encoder_length,
    prediction_length=prediction_length,
    batch_size=batch_size,
    num_workers=num_workers,
    pin_memory=False if pin_memory is None else pin_memory,
    persistent_workers=(
        False if persistent_workers is None else persistent_workers
    ),
)

train_config = build_default_train_config(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=devices,
    precision=precision,
    enable_progress_bar=(
        True if enable_progress_bar is None else enable_progress_bar
    ),
    default_root_dir=output_dir,
)
_apply_early_apple_silicon_environment_defaults(
    requested_device_profile=device_profile,
    train_config=train_config,
)

observability_config = build_default_observability_config(
    output_dir=output_dir,
    enable_device_stats=(
        True if enable_device_stats is None else enable_device_stats
    ),
    enable_rich_progress_bar=(
        True if enable_rich_progress_bar is None else enable_rich_progress_bar
    ),
)

snapshot_config = build_default_snapshot_config(output_dir=output_dir)
runtime_environment = detect_runtime_environment()
profile_resolution = resolve_device_profile(
    requested_profile=device_profile,
    environment=runtime_environment,
    train_config=train_config,
    data_config=config.data,
    observability_config=observability_config,
)
config = Config(
    data=profile_resolution.data_config,
    tft=config.tft,
    tcn=config.tcn,
)
train_config = profile_resolution.train_config
observability_config = profile_resolution.observability_config
preflight_diagnostics = collect_runtime_diagnostics(
    requested_profile=profile_resolution.requested_profile,
    resolved_profile=profile_resolution.resolved_profile,
    environment=runtime_environment,
    train_config=train_config,
    data_config=config.data,
    observability_config=observability_config,
)

# Display the resolved runtime state so the notebook user can sanity-check
# the profile decision before starting a longer run.
{
    "requested_profile": profile_resolution.requested_profile,
    "resolved_profile": profile_resolution.resolved_profile,
    "applied_defaults": profile_resolution.applied_defaults,
    "diagnostics": format_runtime_diagnostics(preflight_diagnostics),
}


In [ ]:
# Run the shared end-to-end workflow.
#
# Purpose:
# launch the same reusable train/evaluate/predict path used by the script
# entrypoint.
#
# Context:
# this cell intentionally stays thin. It delegates orchestration to the
# same top-level facade that `main.py` exposes, which keeps notebook and
# script behavior aligned even though the heavier implementation now lives
# under `src/workflows/`.
artifacts = run_training_workflow(
    config,
    train_config=train_config,
    snapshot_config=snapshot_config,
    observability_config=observability_config,
    requested_device_profile=profile_resolution.requested_profile,
    resolved_device_profile=profile_resolution.resolved_profile,
    applied_profile_defaults=profile_resolution.applied_defaults,
    runtime_environment=runtime_environment,
    preflight_diagnostics=preflight_diagnostics,
    fail_on_preflight_errors=fail_on_preflight_errors,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    optimizer_name=optimizer_name,
    output_dir=output_dir,
    seed=seed,
    skip_test=skip_test,
    skip_predict=skip_predict,
    save_predictions=save_predictions,
)


# Show the compact JSON-friendly summary captured for this run.
artifacts.summary


In [ ]:
# Held-out test metrics, if a test split was available and `skip_test`
# remained False for this run.
artifacts.test_metrics

In [ ]:
# Inspect one raw prediction batch.
#
# Context:
# the workflow intentionally leaves saved prediction tensors in direct
# model-output form so later notebook cells can decide how they want to
# aggregate, visualize, or calibrate them.
if artifacts.test_predictions:
    first_batch = artifacts.test_predictions[0]
    print(first_batch.shape)
    first_batch
else:
    print("No test predictions were produced for this run.")
